# 9.1 想讓模型學到哪些行為？


問盒中幾球，沒給圖片或數量。編出3沒有依據，只說不回答又無法往下走；「目前看不到數量，請提供球數或圖片」既說明缺項，也幫人繼續。本章分別教有用、誠實與安全。

有用看是否推進目標，誠實看是否符合可見依據，安全看是否守任務明定邊界。下面一筆各自標記；另一筆兩回答都越界，只有相對危害大小不同。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
record = {
    "提問": "盒中有幾顆球？",
    "回答": "目前沒有數量資訊，請提供球數。",
    "有用": True,
    "誠實": True,
    "安全": True,
}
pair = {"回答0安全": False, "回答1安全": False, "相對較安全": 0}
print(record["有用"], record["誠實"], record["安全"])
print("較安全回答可直接當安全正例", pair[f"回答{pair['相對較安全']}安全"])

第一筆輸入是人工判斷過的例子，輸出`True True True`。第二筆故意表示兩個回答都不符合安全邊界，但比較之下回答0危害較小。最後一行中的`f"..."`會把大括號內的運算結果插入文字：先用`pair['相對較安全']`讀出編號0，再組成欄位名稱「回答0安全」，最後用這個名稱查字典，結果是`False`。內外引號分別用單引號與雙引號，讓欄位名稱的引號不會提前結束外層文字。這行的目的，是防止把「兩者中較安全」誤寫成「本身安全」。例如兩個回答都洩漏他人的祕密，只是洩漏程度不同，排名不能把較小的洩漏變成可接受正例。

偏好標籤問哪個回答相對更好，安全標籤問各自是否通過規則，兩者不能互相代替。下節先看可見資訊是否足夠，再處理錯誤前提和拒絕。

練習把第二筆的「回答0安全」改成`True`，先預測最後一行會變`True`，再重跑核對。接著口頭說明：有用與誠實是否也會自動變成True？答案是不會，程式也沒有提供那兩項證據。後面的各節將分別處理缺資訊、錯誤前提與拒絕邊界，避免用一個總標籤掩蓋不同需求。

<details>
<summary>補充：實作約定與原始紀錄</summary>

這個差別在使用外部偏好資料時尤其重要。偏好標籤回答「哪個相對更好」，安全標籤回答「各自是否通過規則」，它們處理的問題不同。例如PKU-SafeRLHF是保存回答偏好與各自安全標籤的外部資料集；若採用這類資料，要保留來源原本的安全欄位，並遵守其非商用授權，不能只留下勝者就丟掉其餘資訊。這段程式沒有載入任何外部資料，只示範欄位應如何被解讀。

若裁短外部提問或回答，原本的安全標籤也要重新檢查，因為片段可能失去重要上下文。來源與可發布範圍見[資料來源說明](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/release-licenses.md)。

短程式沿用本課工具與原計算語義。安裝、長訓練與重做操作見[訓練配方](https://birdhackor.github.io/tiny-perceptron-vlm/training.html)，不需要先完成長配方才能閱讀這個例子。

</details>
